In [57]:
# Cell 32 — Memory-safe pair similarity

from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import vstack

# Fit TF-IDF on unique clause texts only
unique_texts = pd.concat([
    clause_pairs_sample["text_a"],
    clause_pairs_sample["text_b"]
]).astype(str).drop_duplicates()

pair_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    max_features=10000,
    sublinear_tf=True
)

pair_vectorizer.fit(unique_texts)

# Transform A and B separately
a_matrix = pair_vectorizer.transform(
    clause_pairs_sample["text_a"].astype(str)
)

b_matrix = pair_vectorizer.transform(
    clause_pairs_sample["text_b"].astype(str)
)

# Compute ONLY corresponding row-to-row similarities
numerator = a_matrix.multiply(b_matrix).sum(axis=1)

a_norm = np.sqrt(
    a_matrix.multiply(a_matrix).sum(axis=1)
)

b_norm = np.sqrt(
    b_matrix.multiply(b_matrix).sum(axis=1)
)

denominator = np.asarray(
    a_norm
).ravel() * np.asarray(
    b_norm
).ravel()

pair_similarities = np.divide(
    np.asarray(numerator).ravel(),
    denominator,
    out=np.zeros_like(denominator, dtype=float),
    where=denominator != 0
)

clause_pairs_sample["semantic_similarity"] = pair_similarities

print(
    clause_pairs_sample["semantic_similarity"].describe()
)

count    42838.000000
mean         0.057630
std          0.063791
min          0.000000
25%          0.019551
50%          0.037849
75%          0.073036
max          1.000000
Name: semantic_similarity, dtype: float64


In [58]:
# Cell 33 — Candidate conflict pairs

candidate_conflicts = clause_pairs_sample[
    clause_pairs_sample["semantic_similarity"] >= 0.15
].copy()

print("Total clause pairs:", len(clause_pairs_sample))
print("Potentially related pairs:", len(candidate_conflicts))

display(
    candidate_conflicts[
        [
            "document_id",
            "span_a",
            "span_b",
            "semantic_similarity",
            "modality_a",
            "modality_b",
            "text_a",
            "text_b"
        ]
    ]
    .sort_values(
        "semantic_similarity",
        ascending=False
    )
    .head(10)
)

Total clause pairs: 42838
Potentially related pairs: 2982


,document_id,span_a,span_b,semantic_similarity,modality_a,modality_b,text_a,text_b
8113,245,18,26,1.000000,obligation,obligation,2) a single copy of all Confidential Informati...,2) a single copy of all Confidential Informati...
8086,245,17,25,1.000000,obligation,obligation,1) neither party will be required to delete el...,1) neither party will be required to delete el...
10538,305,8,10,1.000000,obligation,obligation,"As used in this Agreement, the “Representative...","As used in this Agreement, the “Representative..."
35431,622,69,155,1.000000,obligation,obligation,Evolent shall cooperate with ABCO and provide ...,Evolent shall cooperate with ABCO and provide ...
21879,478,14,55,0.968973,obligation,obligation,The Receiving Party shall ensure that all copi...,The Receiving Party shall ensure that all copi...
22738,53,45,48,0.915690,obligation,obligation,6. The ETI shall be entitled to disclose or ma...,8. Each Respondent shall be entitled to disclo...
38024,622,159,160,0.906407,permission,permission,8.1 Confidential Information: ABCO acknowledge...,Evolent acknowledges that in connection with i...
19209,439,32,48,0.896548,obligation,obligation,Section 6 shall survive termination or expirat...,This provision shall survive the termination o...
11459,319,37,48,0.893444,obligation,prohibition,The Company shall:,The Company shall not:
18133,421,63,65,0.885923,prohibition,prohibition,"Master Distributor shall not, during the term ...","Master Distributor shall not, during the term ..."


In [59]:
import re
import numpy as np
import pandas as pd

def normalize_clause(text):
    text = str(text).lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def extract_action_core(text):
    """
    Remove common modal words so that we can compare
    the underlying action/content of two clauses.
    """
    text = normalize_clause(text)

    modal_words = [
        "shall not", "must not", "may not",
        "will not", "cannot",
        "shall", "must", "may", "will"
    ]

    for modal in modal_words:
        text = re.sub(r'\b' + re.escape(modal) + r'\b', ' ', text)

    text = re.sub(r'\s+', ' ', text).strip()
    return text


def modality_conflict(mod_a, mod_b):
    if {mod_a, mod_b} == {"obligation", "prohibition"}:
        return 1
    
    if {mod_a, mod_b} == {"permission", "prohibition"}:
        return 1
    
    return 0


def explicit_negation_conflict(text_a, text_b):
    a = normalize_clause(text_a)
    b = normalize_clause(text_b)

    # Positive vs explicit negative construction
    patterns_positive = [
        r'\bshall\b',
        r'\bmust\b',
        r'\bmay\b',
        r'\bwill\b'
    ]

    patterns_negative = [
        r'\bshall not\b',
        r'\bmust not\b',
        r'\bmay not\b',
        r'\bwill not\b',
        r'\bcannot\b'
    ]

    a_negative = any(re.search(p, a) for p in patterns_negative)
    b_negative = any(re.search(p, b) for p in patterns_negative)

    a_positive = any(re.search(p, a) for p in patterns_positive) and not a_negative
    b_positive = any(re.search(p, b) for p in patterns_positive) and not b_negative

    return int((a_positive and b_negative) or (b_positive and a_negative))

In [60]:
candidate_conflicts = candidate_conflicts.copy()

candidate_conflicts["modality_conflict"] = candidate_conflicts.apply(
    lambda r: modality_conflict(
        r["modality_a"],
        r["modality_b"]
    ),
    axis=1
)

candidate_conflicts["negation_conflict"] = candidate_conflicts.apply(
    lambda r: explicit_negation_conflict(
        r["text_a"],
        r["text_b"]
    ),
    axis=1
)

candidate_conflicts["action_a"] = candidate_conflicts["text_a"].apply(
    extract_action_core
)

candidate_conflicts["action_b"] = candidate_conflicts["text_b"].apply(
    extract_action_core
)

print("Candidate pairs:", len(candidate_conflicts))

print("\nModality conflicts:")
print(candidate_conflicts["modality_conflict"].value_counts())

print("\nNegation conflicts:")
print(candidate_conflicts["negation_conflict"].value_counts())

Candidate pairs: 2982

Modality conflicts:
modality_conflict
0    2005
1     977
Name: count, dtype: int64

Negation conflicts:
negation_conflict
0    2069
1     913
Name: count, dtype: int64


In [61]:
strong_conflicts = candidate_conflicts[
    (
        (candidate_conflicts["modality_conflict"] == 1) |
        (candidate_conflicts["negation_conflict"] == 1)
    )
    &
    (candidate_conflicts["semantic_similarity"] >= 0.30)
].copy()

strong_conflicts = strong_conflicts.sort_values(
    ["semantic_similarity", "modality_conflict", "negation_conflict"],
    ascending=False
)

print("Strong conflict candidates:", len(strong_conflicts))

display(
    strong_conflicts[
        [
            "document_id",
            "span_a",
            "span_b",
            "semantic_similarity",
            "modality_a",
            "modality_b",
            "modality_conflict",
            "negation_conflict",
            "text_a",
            "text_b"
        ]
    ].head(20)
)

Strong conflict candidates: 115


,document_id,span_a,span_b,semantic_similarity,modality_a,modality_b,modality_conflict,negation_conflict,text_a,text_b
11459,319,37,48,0.893444,obligation,prohibition,1,1,The Company shall:,The Company shall not:
3559,170,63,64,0.647515,prohibition,obligation,1,1,The Receiving Party shall not be entitled to c...,"2.All samples, models, computer programs, draw..."
14708,382,31,33,0.638032,prohibition,permission,1,1,With CMC's prior written consent (which will n...,"Valco also may disclose, on a need to know bas..."
11008,305,37,74,0.624973,obligation,prohibition,1,1,"For purposes of this Agreement, the Open Mobil...",The Open Mobile Alliance will exercise reasona...
14694,382,29,31,0.622248,permission,prohibition,1,1,"(c) Valco may disclose, on a need to know basi...",With CMC's prior written consent (which will n...
33624,607,39,49,0.600768,prohibition,obligation,1,1,(i) the obligations set forth in the second an...,Upon termination of this Agreement or the requ...
11033,305,38,75,0.529865,obligation,prohibition,1,1,The Open Mobile Alliance’s duties under this A...,Participant agrees that the Open Mobile Allian...
3759,170,91,92,0.529474,prohibition,permission,1,1,2. The Disclosing Party shall not have any lia...,The Receiving Party agrees to indemnify the Di...
31299,591,109,111,0.493976,permission,prohibition,1,1,(i) consents to the continued representation o...,"Notwithstanding the foregoing, in the event of..."
11009,305,37,75,0.485297,obligation,prohibition,1,1,"For purposes of this Agreement, the Open Mobil...",Participant agrees that the Open Mobile Allian...


In [62]:
def clean_proposition(text):
    text = normalize_clause(text)

    # Remove common modal constructions
    text = re.sub(
        r'\b(?:shall|shall not|must|must not|may|may not|will|will not|'
        r'cannot|is required to|are required to|is permitted to|'
        r'are permitted to)\b',
        ' ',
        text
    )

    # Remove common legal filler
    text = re.sub(
        r'\b(?:the|a|an|any|such|herein|thereof|thereto|'
        r'provided that|notwithstanding the foregoing)\b',
        ' ',
        text
    )

    text = re.sub(r'\s+', ' ', text).strip()

    return text


candidate_conflicts["proposition_a"] = candidate_conflicts["text_a"].apply(
    clean_proposition
)

candidate_conflicts["proposition_b"] = candidate_conflicts["text_b"].apply(
    clean_proposition
)

display(
    candidate_conflicts[
        [
            "document_id",
            "semantic_similarity",
            "modality_a",
            "modality_b",
            "proposition_a",
            "proposition_b"
        ]
    ].head(15)
)

,document_id,semantic_similarity,modality_a,modality_b,proposition_a,proposition_b
0,113,0.152532,obligation,prohibition,"whereas, in connection with business relations...",(ii) was or becomes available to recipient on ...
2,113,0.213030,obligation,obligation,"whereas, in connection with business relations...",recipient further agrees to disclose confident...
4,113,0.154526,obligation,obligation,"whereas, in connection with business relations...","4. in addition, without prior written consent ..."
12,113,0.181934,obligation,obligation,"whereas, in connection with business relations...",recipient’s obligations of confidentiality and...
18,113,0.529707,prohibition,prohibition,(ii) was or becomes available to recipient on ...,(iii) was within recipient’s possession prior ...
22,113,0.197561,prohibition,obligation,(ii) was or becomes available to recipient on ...,each recipient agrees that neither phcs nor of...
45,113,0.176253,prohibition,obligation,(iii) was within recipient’s possession prior ...,recipient’s obligations of confidentiality and...
53,113,0.166255,obligation,obligation,recipient further agrees to disclose confident...,each recipient agrees that neither phcs nor of...
67,113,0.198964,obligation,obligation,recipient be responsible for improper use of c...,each recipient agrees that neither phcs nor of...
80,113,0.184158,obligation,obligation,"4. in addition, without prior written consent ...",each recipient agrees that neither phcs nor of...


In [63]:
prop_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    max_features=5000,
    sublinear_tf=True
)

prop_texts = pd.concat([
    candidate_conflicts["proposition_a"],
    candidate_conflicts["proposition_b"]
]).astype(str)

prop_vectorizer.fit(prop_texts)

prop_a_matrix = prop_vectorizer.transform(
    candidate_conflicts["proposition_a"].astype(str)
)

prop_b_matrix = prop_vectorizer.transform(
    candidate_conflicts["proposition_b"].astype(str)
)

numerator = prop_a_matrix.multiply(prop_b_matrix).sum(axis=1)

norm_a = np.sqrt(
    prop_a_matrix.multiply(prop_a_matrix).sum(axis=1)
)

norm_b = np.sqrt(
    prop_b_matrix.multiply(prop_b_matrix).sum(axis=1)
)

denominator = (
    np.asarray(norm_a).ravel()
    *
    np.asarray(norm_b).ravel()
)

candidate_conflicts["proposition_similarity"] = np.divide(
    np.asarray(numerator).ravel(),
    denominator,
    out=np.zeros_like(denominator, dtype=float),
    where=denominator != 0
)

print(
    candidate_conflicts["proposition_similarity"].describe()
)

count    2982.000000
mean        0.215125
std         0.118255
min         0.007173
25%         0.143707
50%         0.190315
75%         0.252745
max         1.000000
Name: proposition_similarity, dtype: float64


In [64]:
candidate_conflicts["structured_score"] = (
    0.40 * candidate_conflicts["semantic_similarity"]
    +
    0.35 * candidate_conflicts["proposition_similarity"]
    +
    0.25 * candidate_conflicts["modality_conflict"]
)

ranked_conflicts = candidate_conflicts.sort_values(
    "structured_score",
    ascending=False
).copy()

display(
    ranked_conflicts[
        [
            "document_id",
            "span_a",
            "span_b",
            "semantic_similarity",
            "proposition_similarity",
            "modality_a",
            "modality_b",
            "modality_conflict",
            "negation_conflict",
            "structured_score",
            "text_a",
            "text_b"
        ]
    ].head(25)
)

,document_id,span_a,span_b,semantic_similarity,proposition_similarity,modality_a,modality_b,modality_conflict,negation_conflict,structured_score,text_a,text_b
10538,305,8,10,1.000000,1.000000,obligation,obligation,0,0,0.750000,"As used in this Agreement, the “Representative...","As used in this Agreement, the “Representative..."
35431,622,69,155,1.000000,1.000000,obligation,obligation,0,0,0.750000,Evolent shall cooperate with ABCO and provide ...,Evolent shall cooperate with ABCO and provide ...
8086,245,17,25,1.000000,1.000000,obligation,obligation,0,0,0.750000,1) neither party will be required to delete el...,1) neither party will be required to delete el...
8113,245,18,26,1.000000,1.000000,obligation,obligation,0,0,0.750000,2) a single copy of all Confidential Informati...,2) a single copy of all Confidential Informati...
11459,319,37,48,0.893444,0.395617,obligation,prohibition,1,1,0.745844,The Company shall:,The Company shall not:
21879,478,14,55,0.968973,0.968475,obligation,obligation,0,0,0.726555,The Receiving Party shall ensure that all copi...,The Receiving Party shall ensure that all copi...
14708,382,31,33,0.638032,0.627735,prohibition,permission,1,1,0.724920,With CMC's prior written consent (which will n...,"Valco also may disclose, on a need to know bas..."
14694,382,29,31,0.622248,0.585723,permission,prohibition,1,1,0.703902,"(c) Valco may disclose, on a need to know basi...",With CMC's prior written consent (which will n...
22738,53,45,48,0.915690,0.944426,obligation,obligation,0,0,0.696825,6. The ETI shall be entitled to disclose or ma...,8. Each Respondent shall be entitled to disclo...
19209,439,32,48,0.896548,0.955026,obligation,obligation,0,0,0.692878,Section 6 shall survive termination or expirat...,This provision shall survive the termination o...


In [65]:
def classify_candidate(row):
    sim = row["semantic_similarity"]
    prop_sim = row["proposition_similarity"]
    modality = row["modality_conflict"]
    negation = row["negation_conflict"]

    # Very similar propositions with opposing modality
    if (
        modality == 1
        and prop_sim >= 0.45
        and sim >= 0.30
    ):
        return "likely_conflict"

    # Strong lexical/semantic relationship but insufficient
    # proposition alignment
    if (
        (modality == 1 or negation == 1)
        and prop_sim >= 0.20
        and sim >= 0.25
    ):
        return "needs_review"

    return "likely_non_conflict"


ranked_conflicts["structured_class"] = ranked_conflicts.apply(
    classify_candidate,
    axis=1
)

print(
    ranked_conflicts["structured_class"].value_counts()
)

display(
    ranked_conflicts[
        ranked_conflicts["structured_class"] == "likely_conflict"
    ][
        [
            "document_id",
            "span_a",
            "span_b",
            "semantic_similarity",
            "proposition_similarity",
            "modality_a",
            "modality_b",
            "text_a",
            "text_b"
        ]
    ].head(30)
)

structured_class
likely_non_conflict    2806
needs_review            162
likely_conflict          14
Name: count, dtype: int64


,document_id,span_a,span_b,semantic_similarity,proposition_similarity,modality_a,modality_b,text_a,text_b
14708,382,31,33,0.638032,0.627735,prohibition,permission,With CMC's prior written consent (which will n...,"Valco also may disclose, on a need to know bas..."
14694,382,29,31,0.622248,0.585723,permission,prohibition,"(c) Valco may disclose, on a need to know basi...",With CMC's prior written consent (which will n...
3559,170,63,64,0.647515,0.518486,prohibition,obligation,The Receiving Party shall not be entitled to c...,"2.All samples, models, computer programs, draw..."
11008,305,37,74,0.624973,0.491828,obligation,prohibition,"For purposes of this Agreement, the Open Mobil...",The Open Mobile Alliance will exercise reasona...
33624,607,39,49,0.600768,0.504112,prohibition,obligation,(i) the obligations set forth in the second an...,Upon termination of this Agreement or the requ...
3759,170,91,92,0.529474,0.528388,prohibition,permission,2. The Disclosing Party shall not have any lia...,The Receiving Party agrees to indemnify the Di...
23419,534,39,40,0.431765,0.605845,prohibition,obligation,If any Evaluation Material (including Evaluati...,All Evaluation Material provided to you that i...
22963,534,8,11,0.479796,0.537567,permission,prohibition,(b) subject to the section captioned “Legally ...,Subject to the section captioned “Legally Requ...
31048,591,73,74,0.403886,0.613440,prohibition,obligation,If any Evaluation Material includes materials ...,All Evaluation Material that is entitled to pr...
20889,464,42,43,0.372296,0.552928,prohibition,obligation,10. To the extent that any Confidential Inform...,All Confidential Information that is entitled ...


In [66]:
# Make sure the condition extractor exists

CONDITION_PATTERNS = [
    r"\bif\b[^,.;]*",
    r"\bunless\b[^,.;]*",
    r"\bprovided that\b[^,.;]*",
    r"\bsubject to\b[^,.;]*",
    r"\bin the event that\b[^,.;]*",
    r"\bwhere\b[^,.;]*",
    r"\bonly if\b[^,.;]*",
    r"\bexcept\b[^,.;]*",
    r"\bexcept that\b[^,.;]*",
    r"\bfor\b[^,.;]*transactions",
    r"\bwith respect to\b[^,.;]*"
]

def extract_conditions(text):
    if not isinstance(text, str):
        text = str(text)

    found = []

    for pattern in CONDITION_PATTERNS:
        matches = re.findall(
            pattern,
            text,
            flags=re.IGNORECASE
        )
        found.extend(matches)

    return list(dict.fromkeys(
        x.lower().strip()
        for x in found
    ))


candidate_conflicts["conditions_a"] = candidate_conflicts["text_a"].apply(
    extract_conditions
)

candidate_conflicts["conditions_b"] = candidate_conflicts["text_b"].apply(
    extract_conditions
)

candidate_conflicts["has_condition_a"] = (
    candidate_conflicts["conditions_a"].apply(len) > 0
).astype(int)

candidate_conflicts["has_condition_b"] = (
    candidate_conflicts["conditions_b"].apply(len) > 0
).astype(int)

print(
    "Clause A with conditions:",
    candidate_conflicts["has_condition_a"].sum()
)

print(
    "Clause B with conditions:",
    candidate_conflicts["has_condition_b"].sum()
)

Clause A with conditions: 959
Clause B with conditions: 1008


In [67]:
def condition_similarity(row):
    a = set(row["conditions_a"])
    b = set(row["conditions_b"])

    # Neither clause has a condition
    if not a and not b:
        return 1.0

    # One has condition and the other doesn't
    if not a or not b:
        return 0.0

    # Jaccard similarity
    return len(a & b) / len(a | b)


candidate_conflicts["condition_similarity"] = (
    candidate_conflicts.apply(
        condition_similarity,
        axis=1
    )
)

print(
    candidate_conflicts["condition_similarity"].describe()
)

count    2982.000000
mean        0.480410
std         0.499562
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max         1.000000
Name: condition_similarity, dtype: float64


In [68]:
def party_similarity(row):
    a = set(row.get("parties_a", []))
    b = set(row.get("parties_b", []))

    if not a and not b:
        return 1.0

    if not a or not b:
        return 0.0

    return len(a & b) / len(a | b)


# Handle either parties_a / parties_b or recreate them
if "parties_a" not in candidate_conflicts.columns:
    candidate_conflicts["parties_a"] = candidate_conflicts["text_a"].apply(
        extract_parties
    )

if "parties_b" not in candidate_conflicts.columns:
    candidate_conflicts["parties_b"] = candidate_conflicts["text_b"].apply(
        extract_parties
    )

candidate_conflicts["party_similarity"] = (
    candidate_conflicts.apply(
        party_similarity,
        axis=1
    )
)

print(
    candidate_conflicts["party_similarity"].describe()
)

count    2982.000000
mean        0.652364
std         0.433686
min         0.000000
25%         0.000000
50%         1.000000
75%         1.000000
max         1.000000
Name: party_similarity, dtype: float64


In [69]:
candidate_conflicts["final_structured_score"] = (
    0.25 * candidate_conflicts["semantic_similarity"]
    +
    0.30 * candidate_conflicts["proposition_similarity"]
    +
    0.20 * candidate_conflicts["modality_conflict"]
    +
    0.15 * candidate_conflicts["condition_similarity"]
    +
    0.10 * candidate_conflicts["party_similarity"]
)

ranked_structured = candidate_conflicts.sort_values(
    "final_structured_score",
    ascending=False
).copy()

display(
    ranked_structured[
        [
            "document_id",
            "span_a",
            "span_b",
            "semantic_similarity",
            "proposition_similarity",
            "modality_a",
            "modality_b",
            "modality_conflict",
            "condition_similarity",
            "party_similarity",
            "final_structured_score",
            "text_a",
            "text_b"
        ]
    ].head(25)
)

,document_id,span_a,span_b,semantic_similarity,proposition_similarity,modality_a,modality_b,modality_conflict,condition_similarity,party_similarity,final_structured_score,text_a,text_b
10538,305,8,10,1.000000,1.000000,obligation,obligation,0,1.0,1.000000,0.800000,"As used in this Agreement, the “Representative...","As used in this Agreement, the “Representative..."
8086,245,17,25,1.000000,1.000000,obligation,obligation,0,1.0,1.000000,0.800000,1) neither party will be required to delete el...,1) neither party will be required to delete el...
8113,245,18,26,1.000000,1.000000,obligation,obligation,0,1.0,1.000000,0.800000,2) a single copy of all Confidential Informati...,2) a single copy of all Confidential Informati...
35431,622,69,155,1.000000,1.000000,obligation,obligation,0,1.0,1.000000,0.800000,Evolent shall cooperate with ABCO and provide ...,Evolent shall cooperate with ABCO and provide ...
14708,382,31,33,0.638032,0.627735,prohibition,permission,1,1.0,1.000000,0.797828,With CMC's prior written consent (which will n...,"Valco also may disclose, on a need to know bas..."
11459,319,37,48,0.893444,0.395617,obligation,prohibition,1,1.0,1.000000,0.792046,The Company shall:,The Company shall not:
21879,478,14,55,0.968973,0.968475,obligation,obligation,0,1.0,1.000000,0.782786,The Receiving Party shall ensure that all copi...,The Receiving Party shall ensure that all copi...
14694,382,29,31,0.622248,0.585723,permission,prohibition,1,1.0,1.000000,0.781279,"(c) Valco may disclose, on a need to know basi...",With CMC's prior written consent (which will n...
22738,53,45,48,0.915690,0.944426,obligation,obligation,0,1.0,1.000000,0.762250,6. The ETI shall be entitled to disclose or ma...,8. Each Respondent shall be entitled to disclo...
19209,439,32,48,0.896548,0.955026,obligation,obligation,0,1.0,1.000000,0.760645,Section 6 shall survive termination or expirat...,This provision shall survive the termination o...


In [70]:
def final_conflict_class(row):

    # Strong proposition alignment + opposing modality
    if (
        row["modality_conflict"] == 1
        and row["proposition_similarity"] >= 0.45
        and row["semantic_similarity"] >= 0.30
        and row["condition_similarity"] >= 0.50
    ):
        return "likely_conflict"

    # Related clauses but insufficient evidence
    if (
        (
            row["modality_conflict"] == 1
            or row["negation_conflict"] == 1
        )
        and row["proposition_similarity"] >= 0.20
        and row["semantic_similarity"] >= 0.25
    ):
        return "needs_review"

    return "likely_non_conflict"


ranked_structured["final_class"] = (
    ranked_structured.apply(
        final_conflict_class,
        axis=1
    )
)

print(
    ranked_structured["final_class"].value_counts()
)

final_class
likely_non_conflict    2806
needs_review            171
likely_conflict           5
Name: count, dtype: int64


In [71]:
final_candidates = ranked_structured[
    ranked_structured["final_class"] == "likely_conflict"
].copy()

print(
    "Final likely conflicts:",
    len(final_candidates)
)

display(
    final_candidates[
        [
            "document_id",
            "span_a",
            "span_b",
            "semantic_similarity",
            "proposition_similarity",
            "condition_similarity",
            "party_similarity",
            "modality_a",
            "modality_b",
            "final_structured_score",
            "text_a",
            "text_b"
        ]
    ]
)

Final likely conflicts: 5


,document_id,span_a,span_b,semantic_similarity,proposition_similarity,condition_similarity,party_similarity,modality_a,modality_b,final_structured_score,text_a,text_b
14708,382,31,33,0.638032,0.627735,1.0,1.000000,prohibition,permission,0.797828,With CMC's prior written consent (which will n...,"Valco also may disclose, on a need to know bas..."
14694,382,29,31,0.622248,0.585723,1.0,1.000000,permission,prohibition,0.781279,"(c) Valco may disclose, on a need to know basi...",With CMC's prior written consent (which will n...
22963,534,8,11,0.479796,0.537567,1.0,1.000000,permission,prohibition,0.731219,(b) subject to the section captioned “Legally ...,Subject to the section captioned “Legally Requ...
33624,607,39,49,0.600768,0.504112,1.0,0.666667,prohibition,obligation,0.718092,(i) the obligations set forth in the second an...,Upon termination of this Agreement or the requ...
4163,174,83,87,0.329813,0.480459,1.0,1.000000,prohibition,obligation,0.676591,"Any illegality, unenforceability or invalidity...",This Agreement will remain in full force and e...


In [73]:
# Ground-truth contradiction documents

dev_contradictions = dev_df[
    dev_df["target"] == 2
].copy()

test_contradictions = test_df[
    test_df["target"] == 2
].copy()

print("Dev contradiction examples:", len(dev_contradictions))
print("Test contradiction examples:", len(test_contradictions))

print("\nDev label distribution:")
print(dev_df["target"].value_counts().sort_index())

print("\nTest label distribution:")
print(test_df["target"].value_counts().sort_index())

Dev contradiction examples: 95
Test contradiction examples: 220

Dev label distribution:
target
0    423
1    519
2     95
Name: count, dtype: int64

Test label distribution:
target
0    903
1    968
2    220
Name: count, dtype: int64


In [74]:
def get_gold_evidence_ids(document, hypothesis):
    """
    Return gold evidence span IDs for a document/hypothesis pair.
    Used ONLY for evaluation.
    """

    for annotation_set in document.get("annotation_sets", []):
        annotations = annotation_set.get("annotations", {})

        for hypothesis_id, annotation in annotations.items():

            # Match using the hypothesis ID when available
            if str(hypothesis_id) == str(hypothesis):
                return annotation.get("spans", [])

    return []


# Better approach: use the row's hypothesis ID if your ml_df contains it.
print(dev_df.columns.tolist())

['document_id', 'label_id', 'short_description', 'hypothesis', 'choice', 'target', 'document_text', 'split', 'context_1', 'context_3', 'context_5', 'context_10', 'bert_text_1', 'bert_text_3', 'bert_text_5', 'bert_text_10']


In [75]:
# Check the hypothesis IDs available in the original ContractNLI data

print("Unique label IDs in dev:", sorted(dev_df["label_id"].unique()))
print("Unique label IDs in test:", sorted(test_df["label_id"].unique()))

print("\nNumber of unique hypotheses:", dev_df["label_id"].nunique())

display(
    dev_df[
        ["label_id", "short_description", "hypothesis"]
    ].drop_duplicates("label_id").sort_values("label_id")
)

Unique label IDs in dev: ['nda-1', 'nda-10', 'nda-11', 'nda-12', 'nda-13', 'nda-15', 'nda-16', 'nda-17', 'nda-18', 'nda-19', 'nda-2', 'nda-20', 'nda-3', 'nda-4', 'nda-5', 'nda-7', 'nda-8']
Unique label IDs in test: ['nda-1', 'nda-10', 'nda-11', 'nda-12', 'nda-13', 'nda-15', 'nda-16', 'nda-17', 'nda-18', 'nda-19', 'nda-2', 'nda-20', 'nda-3', 'nda-4', 'nda-5', 'nda-7', 'nda-8']

Number of unique hypotheses: 17


,label_id,short_description,hypothesis
5,nda-1,Explicit identification,All Confidential Information shall be expressl...
3,nda-10,Confidentiality of Agreement,Receiving Party shall not disclose the fact th...
0,nda-11,No reverse engineering,Receiving Party shall not reverse engineer any...
7,nda-12,Permissible development of similar information,Receiving Party may independently develop info...
14,nda-13,Permissible acquirement of similar information,Receiving Party may acquire information simila...
2,nda-15,No licensing,Agreement shall not grant Receiving Party any ...
1,nda-16,Return of confidential information,Receiving Party shall destroy or return some C...
12,nda-17,Permissible copy,Receiving Party may create a copy of some Conf...
10,nda-18,No solicitation,Receiving Party shall not solicit some of Disc...
6,nda-19,Survival of obligations,Some obligations of Agreement may survive term...


In [79]:
def get_gold_spans(document_id, label_id):
    """
    Get ContractNLI gold evidence span IDs.

    IMPORTANT:
    This function is used only for evaluation.
    Gold evidence is NOT supplied to the model.
    """

    document = document_map[str(document_id)]

    for annotation_set in document.get("annotation_sets", []):
        annotations = annotation_set.get("annotations", {})

        annotation = annotations.get(str(label_id))

        if annotation is not None:
            return annotation.get("spans", [])

    return []


# Test on one example
sample_row = dev_df.iloc[0]

gold = get_gold_spans(
    sample_row["document_id"],
    sample_row["label_id"]
)

print("Document:", sample_row["document_id"])
print("Label ID:", sample_row["label_id"])
print("Gold evidence spans:", gold)

Document: 3
Label ID: nda-11
Gold evidence spans: []


In [77]:
def normalize_span_ids(spans):
    return {
        int(s)
        for s in spans
        if str(s).isdigit()
    }


def retrieval_recall(df, top_k=5):

    hits = 0
    total = 0

    for _, row in df.iterrows():

        gold_spans = normalize_span_ids(
            get_gold_spans(
                row["document_id"],
                row["label_id"]
            )
        )

        # NotMentioned examples normally have no evidence
        if not gold_spans:
            continue

        retrieved = retrieve_clauses(
            row["document_id"],
            row["hypothesis"],
            top_k=top_k
        )

        retrieved_spans = set(
            retrieved["span_id"].astype(int)
        )

        if gold_spans.intersection(retrieved_spans):
            hits += 1

        total += 1

    return hits / total if total else 0


dev_recall = retrieval_recall(
    dev_df,
    top_k=5
)

test_recall = retrieval_recall(
    test_df,
    top_k=5
)

print(f"Dev Retrieval Recall@5 : {dev_recall:.4f}")
print(f"Test Retrieval Recall@5: {test_recall:.4f}")

Dev Retrieval Recall@5 : 0.0000
Test Retrieval Recall@5: 0.0000


In [78]:
for k in [1, 3, 5, 10]:

    dev_r = retrieval_recall(
        dev_df,
        top_k=k
    )

    test_r = retrieval_recall(
        test_df,
        top_k=k
    )

    print(
        f"Recall@{k:<2} | "
        f"Dev: {dev_r:.4f} | "
        f"Test: {test_r:.4f}"
    )

Recall@1  | Dev: 0.0000 | Test: 0.0000
Recall@3  | Dev: 0.0000 | Test: 0.0000
Recall@5  | Dev: 0.0000 | Test: 0.0000
Recall@10 | Dev: 0.0000 | Test: 0.0000


In [80]:
row = dev_df.iloc[0]

print("Document ID:", row["document_id"])
print("Label ID:", row["label_id"])
print("Choice:", row["choice"])
print("Target:", row["target"])
print("Hypothesis:", row["hypothesis"])

gold_spans = get_gold_spans(
    row["document_id"],
    row["label_id"]
)

print("\nGold evidence spans:", gold_spans)

Document ID: 3
Label ID: nda-11
Choice: Entailment
Target: 1
Hypothesis: Receiving Party shall not reverse engineer any objects which embody Disclosing Party's Confidential Information.

Gold evidence spans: []


In [81]:
# Inspect the exact raw annotation for document 3
row = dev_df.iloc[0]

doc = document_map[str(row["document_id"])]

print("DataFrame:")
print(" document_id:", row["document_id"])
print(" label_id:", row["label_id"])
print(" choice:", row["choice"])
print(" target:", row["target"])
print(" hypothesis:", row["hypothesis"])

print("\nRAW ANNOTATIONS:")

for annotation_set in doc["annotation_sets"]:
    annotations = annotation_set["annotations"]

    for key, annotation in annotations.items():
        print(
            "\nKEY:", key,
            "\nVALUE:", annotation
        )

DataFrame:
 document_id: 3
 label_id: nda-11
 choice: Entailment
 target: 1
 hypothesis: Receiving Party shall not reverse engineer any objects which embody Disclosing Party's Confidential Information.

RAW ANNOTATIONS:


KeyError: 'annotation_sets'

In [82]:
# Pick one known example
row = dev_df[dev_df["target"] == 1].iloc[0]

doc = document_map[str(row["document_id"])]

print("Document ID:", row["document_id"])
print("Label ID:", row["label_id"])
print("Choice:", row["choice"])
print("Target:", row["target"])

print("\nType of document:")
print(type(doc))

print("\nDocument keys:")
print(doc.keys())

print("\nAnnotation sets object:")
print(type(doc.get("annotation_sets", None)))

print("\nAnnotation sets:")
print(doc.get("annotation_sets", None))

Document ID: 3
Label ID: nda-11
Choice: Entailment
Target: 1

Type of document:
<class 'dict'>

Document keys:
dict_keys(['document', 'split'])

Annotation sets object:
<class 'NoneType'>

Annotation sets:
None


In [83]:
print("Document object:")
print(doc)

print("\n" + "="*80)

for key in doc.keys():
    print(
        f"KEY = {key} | "
        f"TYPE = {type(doc[key])}"
    )

Document object:
{'document': {'id': 3, 'file_name': '09-24-2019-04-25-05-3914910473.pdf', 'text': "OISAIR PROJECT\nTWO-WAY CONFIDENTIALITY AND NON-DISCLOSURE AGREEMENT\n(TO BE SIGNED ELECTRONICALLY THROUGH THE INNOVAIR PLATFORM)\nThis Confidentiality and Non-Disclosure Agreement (hereinafter referred to as the “Agreement”) dated ………………………. (“Effective Date”) is made by and between:\n1) <Research institution name> with registered offices located in ……………………………, Tax registration No ………, represented by ………………………………….., in the legal capacity as …………………….. Hereinafter referred to as “………………..”\n2) <Company name> with registered offices located in ………………………….. Tax registration No. ……………, represented by …………………………., in the legal capacity as ………………………….. Hereinafter referred to as “………………..”\nThe above parties hereinafter collectively referred to as the “Parties” and individually as a “Party”.\nWHEREAS\nThe Parties are willing to exchange Confidential Information in the form of certain scient

In [84]:
print("train_data type:", type(train_data))
print("dev_data type:", type(dev_data))
print("test_data type:", type(test_data))

print("\nFirst dev_data object:")
print(dev_data[0])

print("\nFirst dev_data object type:")
print(type(dev_data[0]))

train_data type: <class 'dict'>
dev_data type: <class 'dict'>
test_data type: <class 'dict'>

First dev_data object:


KeyError: 0

In [85]:
def get_gold_spans(document_id, label_id):
    """
    Retrieve gold ContractNLI evidence spans.

    IMPORTANT:
    Gold evidence is used ONLY for evaluation.
    It is never supplied to the retrieval/reasoning model.
    """

    entry = document_map[str(document_id)]

    # Actual structure:
    # entry["document"]["annotation_sets"][0]["annotations"]

    document = entry["document"]

    for annotation_set in document.get("annotation_sets", []):
        annotations = annotation_set.get("annotations", {})

        annotation = annotations.get(str(label_id))

        if annotation is not None:
            return annotation.get("spans", [])

    return []

In [86]:
# Test the exact example we were debugging

row = dev_df.iloc[0]

print("Document ID:", row["document_id"])
print("Label ID:", row["label_id"])
print("Choice:", row["choice"])
print("Target:", row["target"])
print("Hypothesis:", row["hypothesis"])

gold_spans = get_gold_spans(
    row["document_id"],
    row["label_id"]
)

print("\nGold evidence spans:", gold_spans)

Document ID: 3
Label ID: nda-11
Choice: Entailment
Target: 1
Hypothesis: Receiving Party shall not reverse engineer any objects which embody Disclosing Party's Confidential Information.

Gold evidence spans: [33, 39]


In [87]:
def normalize_span_ids(spans):
    return {
        int(s)
        for s in spans
        if str(s).isdigit()
    }

In [88]:
def retrieval_recall(df, top_k=5):

    hits = 0
    total = 0

    for _, row in df.iterrows():

        gold_spans = normalize_span_ids(
            get_gold_spans(
                row["document_id"],
                row["label_id"]
            )
        )

        # Skip NotMentioned cases with no evidence
        if not gold_spans:
            continue

        retrieved = retrieve_clauses(
            row["document_id"],
            row["hypothesis"],
            top_k=top_k
        )

        retrieved_spans = set(
            retrieved["span_id"].astype(int)
        )

        if gold_spans.intersection(retrieved_spans):
            hits += 1

        total += 1

    return hits / total if total else 0

In [89]:
print("Retrieval evaluation")
print("=" * 45)

for k in [1, 3, 5, 10]:

    dev_r = retrieval_recall(
        dev_df,
        top_k=k
    )

    test_r = retrieval_recall(
        test_df,
        top_k=k
    )

    print(
        f"Recall@{k:<2} | "
        f"Dev: {dev_r:.4f} | "
        f"Test: {test_r:.4f}"
    )

Retrieval evaluation
Recall@1  | Dev: 0.2883 | Test: 0.3114
Recall@3  | Dev: 0.5554 | Test: 0.5673
Recall@5  | Dev: 0.6580 | Test: 0.6726
Recall@10 | Dev: 0.8127 | Test: 0.8316


In [93]:
def build_structured_features(clause_df):
    df = clause_df.copy()

    # retrieve_clauses() returns the clause text in "text"
    df["modality"] = df["text"].apply(detect_modality)
    df["negation"] = df["text"].apply(detect_negation)
    df["conditions"] = df["text"].apply(extract_conditions)
    df["temporal"] = df["text"].apply(extract_temporal)
    df["numbers"] = df["text"].apply(extract_numbers)
    df["parties"] = df["text"].apply(extract_parties)
    df["proposition"] = df["text"].apply(clean_proposition)

    return df

In [95]:
def get_structured_candidates(row, top_k=5):

    retrieved = retrieve_clauses(
        row["document_id"],
        row["hypothesis"],
        top_k=top_k
    )

    if retrieved.empty:
        return []

    structured = build_structured_features(retrieved)

    candidates = []

    for _, clause in structured.iterrows():

        candidates.append({
            "document_id": row["document_id"],
            "label_id": row["label_id"],
            "hypothesis": row["hypothesis"],
            "target": row["target"],
            "span_id": int(clause["span_id"]),
            "clause": clause["text"],   # corrected
            "retrieval_score": float(clause["retrieval_score"]),
            "modality": clause["modality"],
            "negation": clause["negation"],
            "conditions": clause["conditions"],
            "temporal": clause["temporal"],
            "numbers": clause["numbers"],
            "parties": clause["parties"],
            "proposition": clause["proposition"]
        })

    return candidates

In [96]:
test_row = dev_df.iloc[0]

test_candidates = get_structured_candidates(
    test_row,
    top_k=5
)

print("Number of candidates:", len(test_candidates))

display(
    pd.DataFrame(test_candidates)
)

Number of candidates: 5


,document_id,label_id,hypothesis,target,span_id,clause,retrieval_score,modality,negation,conditions,temporal,numbers,parties,proposition
0,3,nda-11,Receiving Party shall not reverse engineer any...,1,39,"f) Not to alter, modify, disassemble, reverse ...",0.222255,none,True,[],[],[],[],"f) not to alter, modify, disassemble, reverse ..."
1,3,nda-11,Receiving Party shall not reverse engineer any...,1,78,<Party>,0.142120,none,False,[],[],[],[party],<party>
2,3,nda-11,Receiving Party shall not reverse engineer any...,1,82,<Party>,0.142120,none,False,[],[],[],[party],<party>
3,3,nda-11,Receiving Party shall not reverse engineer any...,1,56,The Disclosing Party is not obliged to disclos...,0.137541,obligation,True,[],[],[],"[receiving party, disclosing party, party]",disclosing party is not obliged to disclose co...
4,3,nda-11,Receiving Party shall not reverse engineer any...,1,22,“Receiving Party” shall mean the Party that re...,0.135367,obligation,False,[],[],[],"[receiving party, disclosing party, party]",“receiving party” mean party that receives con...


In [97]:
dev_structured_rows = []

for i, (_, row) in enumerate(dev_df.iterrows()):

    candidates = get_structured_candidates(
        row,
        top_k=5
    )

    dev_structured_rows.extend(candidates)

    if (i + 1) % 1000 == 0:
        print(f"Processed {i + 1}/{len(dev_df)} examples")

dev_structured_df = pd.DataFrame(
    dev_structured_rows
)

print("\nDev structured rows:", len(dev_structured_df))
print(
    "Unique Dev documents:",
    dev_structured_df["document_id"].nunique()
)

Processed 1000/1037 examples

Dev structured rows: 5185
Unique Dev documents: 61
